Get API KEY from [Cerebras Website]("https://www.cerebras.ai/"). Save to environment as "CEREBRAS_API_KEY"

In [1]:
SYSTEM_PROMPT_STRING = """
You are an NYT Connections solver.
You will be given 16 words.
OUTPUT ONLY the final groups of words and their categories.
Do NOT provide reasoning or explanations under any circumstances.
DO NOT output any text other than the 4 groups and their categories.
Format exactly like this:

GROUP 1: word1, word2, word3, word4
CATEGORY: category_name

GROUP 2: word1, word2, word3, word4
CATEGORY: category_name

GROUP 3: word1, word2, word3, word4
CATEGORY: category_name

GROUP 4: word1, word2, word3, word4
CATEGORY: category_name
"""

USER_PROMPT = "Here are the 16 words: APPLE, PEAR, TABLE, CHAIR, SOFA, BANANA, DESK, PLUM, COUCH, LAMP, ORANGE, STOOL, BED, GRAPE, SHELF, DRAWER"


In [2]:
import os
from cerebras.cloud.sdk import Cerebras

client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

try:
    response = client.chat.completions.create(
        model="llama3.1-8b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_STRING},
            {"role": "user", "content": USER_PROMPT}
        ],
        temperature=0.1,
        max_tokens=600,
    )

    msg = response.choices[0].message
    output = msg.content or ""
    print(output.strip())

except Exception as e:
    print("ERROR:", e)

GROUP 1: APPLE, PEAR, PLUM, ORANGE
CATEGORY: Types of Fruit

GROUP 2: TABLE, DESK, SHELF, DRAWER
CATEGORY: Furniture with a Flat Surface

GROUP 3: CHAIR, STOOL, SOFA, COUCH
CATEGORY: Types of Seating Furniture

GROUP 4: BANANA, GRAPE, BED, LAMP
CATEGORY: Miscellaneous Items


In [14]:
import re

def call_llm(system_prompt, user_prompt,
                        model = "llama3.1-8b",
                        temperature = 0.1,
                        max_tokens = 600):
    """
    Sends a chat request to the Cerebras API and returns the response content.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        msg = response.choices[0].message
        return (msg.content or "").strip()
    
    except Exception as e:
        return f"ERROR: {e}"

def convert_puzzle_to_prompt(words16):
    word_list_str = ", ".join(words16)
    return f"Here are the 16 words: {word_list_str}"

def parse_response_to_pred_groups(response):
    pattern = r"GROUP \d+: (.+)"
    
    groups = re.findall(pattern, response)
    parsed_groups = [ [word.strip() for word in group.split(",")] for group in groups ]
    
    return parsed_groups

In [3]:
from conn import load_connections_from_hf

hf_split = load_connections_from_hf()
print("puzzles:", len(hf_split))

c:\Users\kunri\miniconda3\envs\cs175\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


puzzles: 652


In [15]:
row0 = hf_split[0]
words16 = row0["words"]
word_list_str = convert_puzzle_to_prompt(words16)
response = call_llm(SYSTEM_PROMPT_STRING, word_list_str)
print(parse_response_to_pred_groups(response))

[['LASER', 'COIL', 'SOLAR PANEL', 'WIND'], ['PLUCK', 'THREAD', 'WRAP', 'SPREADSHEET'], ['WAX', 'HONEYCOMB', 'BALL', 'SPOOL'], ['ORGANISM', 'MOVIE', 'SCHOOL', 'VITAMIN']]


In [7]:
call_llm(SYSTEM_PROMPT_STRING, USER_PROMPT)

'GROUP 1: APPLE, PEAR, PLUM, GRAPE\nCATEGORY: Fruits\n\nGROUP 2: TABLE, DESK, SHELF, DRAWER\nCATEGORY: Furniture\n\nGROUP 3: CHAIR, SOFA, COUCH, BED\nCATEGORY: Furniture\n\nGROUP 4: BANANA, ORANGE, LAMP, STOOL\nCATEGORY: Objects'